# Datathon 7MLET 
## Etapa 7 — Ciclo de Vida MLOps (Tracking de Experimentos com MLflow)

Este notebook registra, no **MLflow** (rodando localmente), os parâmetros e métricas do agente de Thompson Sampling treinado na Etapa 3 — cobrindo o item obrigatório do checklist: *"Tracking de experimentos registrado via ferramenta MLOps (MLflow ou equivalente)"*.

Como o agente já foi treinado e avaliado na Etapa 3, aqui o foco é **registrar** (não re-treinar) o experimento: parâmetros do modelo (priors, braços, contexto), a política final aprendida e as métricas de negócio (taxas de conversão e ganhos sobre os baselines).


In [1]:
import json
import mlflow
import matplotlib.pyplot as plt

print("MLflow version:", mlflow.__version__)


MLflow version: 3.14.0


## 1. Configurando o tracking local

O MLflow grava tudo localmente em um banco **SQLite** (`mlflow.db`), no próprio diretório do projeto — não precisa de nenhum servidor externo para este desafio. (Versões mais recentes do MLflow descontinuaram o backend simples em arquivo `./mlruns`, recomendando um backend de banco de dados — por isso usamos SQLite aqui.) Criamos um **experimento** dedicado para organizar as execuções do bandit.


In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("datathon-bandit-canal-contato")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento ativo:", mlflow.get_experiment_by_name("datathon-bandit-canal-contato"))


2026/07/12 20:28:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/12 20:28:29 INFO mlflow.store.db.utils: Updating database tables


2026/07/12 20:28:30 INFO mlflow.tracking.fluent: Experiment with name 'datathon-bandit-canal-contato' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Experimento ativo: <Experiment: artifact_location='/home/claude/datathon/mlruns/1', creation_time=1783888110848, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783888110848, lifecycle_stage='active', name='datathon-bandit-canal-contato', tags={}, trace_location=None, workspace='default'>


## 2. Carregando os resultados da Etapa 3


In [3]:
with open("etapa3_results.json", "r") as f:
    results = json.load(f)

results


{'context_variable': 'prev_success (poutcome == success)',
 'arms': ['cellular', 'telephone'],
 'best_arm_global_train': 'cellular',
 'legacy_baseline_value': 0.14244287345700607,
 'strong_baseline_value': 0.15076975097952575,
 'thompson_sampling_value': 0.15171916583396977,
 'gain_pp_vs_legacy': 0.9276292376963702,
 'gain_relative_pct_vs_legacy': 6.512289559901063,
 'gain_pp_vs_strong_baseline': 0.0949414854444025,
 'gain_relative_pct_vs_strong_baseline': 0.6297117613286662,
 'ts_learned_policy': {'contexto_0': 'cellular', 'contexto_1': 'telephone'},
 'ts_posterior_params': {'0_cellular': [2345.0, 16701.0],
  '0_telephone': [37.0, 317.0],
  '1_cellular': [406.0, 197.0],
  '1_telephone': [32.0, 12.0]}}

## 3. Registrando parâmetros, métricas e artefatos em uma run do MLflow

- **Parâmetros:** tudo que descreve *como* o modelo foi configurado (algoritmo, braços, contexto, priors, política aprendida) — não muda durante a avaliação, é a "configuração" do experimento.
- **Métricas:** os números que medem *o quão bem* o modelo performou (taxas de conversão, ganhos) — o que a banca/negócio usa para decidir se o modelo é bom o suficiente.
- **Artefatos:** arquivos associados à run — aqui, o `etapa3_results.json` completo e um gráfico comparativo, para rastreabilidade total do experimento.


In [4]:
with mlflow.start_run(run_name="thompson_sampling_contextual_v1") as run:

    # --- Parâmetros do experimento ---
    mlflow.log_param("algoritmo", "Thompson Sampling (Beta-Bernoulli, contextual)")
    mlflow.log_param("braços", results["arms"])
    mlflow.log_param("variável_de_contexto", results["context_variable"])
    mlflow.log_param("prior_alpha", 1.0)
    mlflow.log_param("prior_beta", 1.0)
    mlflow.log_param("melhor_braço_histórico_treino", results["best_arm_global_train"])
    mlflow.log_param("política_aprendida", json.dumps(results["ts_learned_policy"], ensure_ascii=False))
    mlflow.log_param("split_treino_avaliação", "80/20 (hold-out)")
    mlflow.log_param("método_de_avaliação", "replay + estimador estratificado por contexto")

    # --- Métricas de negócio ---
    mlflow.log_metric("conversao_baseline_legado", results["legacy_baseline_value"])
    mlflow.log_metric("conversao_baseline_forte", results["strong_baseline_value"])
    mlflow.log_metric("conversao_thompson_sampling", results["thompson_sampling_value"])
    mlflow.log_metric("ganho_pp_vs_legado", results["gain_pp_vs_legacy"])
    mlflow.log_metric("ganho_relativo_pct_vs_legado", results["gain_relative_pct_vs_legacy"])
    mlflow.log_metric("ganho_pp_vs_baseline_forte", results["gain_pp_vs_strong_baseline"])
    mlflow.log_metric("ganho_relativo_pct_vs_baseline_forte", results["gain_relative_pct_vs_strong_baseline"])

    # Parâmetros posteriores finais (alpha, beta) por contexto/braço, como métricas individuais
    for key, (alpha, beta_) in results["ts_posterior_params"].items():
        mlflow.log_metric(f"posterior_alpha_{key}", alpha)
        mlflow.log_metric(f"posterior_beta_{key}", beta_)
        mlflow.log_metric(f"posterior_media_{key}", alpha / (alpha + beta_))

    # --- Artefato 1: o JSON completo da Etapa 3 ---
    mlflow.log_artifact("etapa3_results.json")

    # --- Artefato 2: gráfico comparativo (reconstruído a partir das métricas) ---
    fig, ax = plt.subplots(figsize=(6, 4))
    politicas = ["Baseline legado", "Baseline forte", "Thompson Sampling"]
    valores = [
        results["legacy_baseline_value"],
        results["strong_baseline_value"],
        results["thompson_sampling_value"],
    ]
    ax.bar(politicas, valores, color=["#B0B0B0", "#8C8C8C", "#55A868"])
    ax.set_ylabel("Taxa de conversão estimada")
    ax.set_title("Comparação de políticas — registrado no MLflow")
    plt.xticks(rotation=15, ha="right")
    for i, v in enumerate(valores):
        ax.annotate(f"{v:.2%}", (i, v), ha="center", va="bottom")
    plt.tight_layout()
    fig.savefig("comparacao_politicas.png", dpi=120)
    plt.close(fig)

    mlflow.log_artifact("comparacao_politicas.png")

    run_id = run.info.run_id
    print(f"Run registrada com sucesso. run_id = {run_id}")


Run registrada com sucesso. run_id = 39c396ee75084da8b0ba3a9f45eab6c2


## 4. Conferindo o que foi registrado

Consultamos de volta a run recém-criada, direto pela API do MLflow, para confirmar que tudo foi salvo corretamente.


In [5]:
client = mlflow.tracking.MlflowClient()
run_info = client.get_run(run_id)

print("=== Parâmetros ===")
for k, v in run_info.data.params.items():
    print(f"  {k}: {v}")

print("\n=== Métricas ===")
for k, v in run_info.data.metrics.items():
    print(f"  {k}: {v:.4f}")

print("\n=== Artefatos ===")
for artifact in client.list_artifacts(run_id):
    print(f"  {artifact.path}")


=== Parâmetros ===
  algoritmo: Thompson Sampling (Beta-Bernoulli, contextual)
  braços: ['cellular', 'telephone']
  variável_de_contexto: prev_success (poutcome == success)
  prior_alpha: 1.0
  prior_beta: 1.0
  melhor_braço_histórico_treino: cellular
  política_aprendida: {"contexto_0": "cellular", "contexto_1": "telephone"}
  split_treino_avaliação: 80/20 (hold-out)
  método_de_avaliação: replay + estimador estratificado por contexto

=== Métricas ===
  conversao_baseline_legado: 0.1424
  conversao_baseline_forte: 0.1508
  conversao_thompson_sampling: 0.1517
  ganho_pp_vs_legado: 0.9276
  ganho_relativo_pct_vs_legado: 6.5123
  ganho_pp_vs_baseline_forte: 0.0949
  ganho_relativo_pct_vs_baseline_forte: 0.6297
  posterior_alpha_0_cellular: 2345.0000
  posterior_beta_0_cellular: 16701.0000
  posterior_media_0_cellular: 0.1231
  posterior_alpha_0_telephone: 37.0000
  posterior_beta_0_telephone: 317.0000
  posterior_media_0_telephone: 0.1045
  posterior_alpha_1_cellular: 406.0000
  poster

## 5. Visualizando no MLflow UI

Para explorar essa (e futuras) execuções de forma visual — comparar métricas entre runs, baixar artefatos, ver gráficos de evolução — basta rodar, no terminal, a partir da pasta do projeto:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

E acessar **http://localhost:5000** no navegador. Lá é possível ver esta run (`thompson_sampling_contextual_v1`), comparar com execuções futuras (ex.: uma versão com mais variáveis de contexto) lado a lado, e baixar o `etapa3_results.json` e o gráfico registrados como artefatos.


## 6. Próximos passos (fora do escopo deste datathon)

- Automatizar o re-treino (ex.: gatilho semanal via cron/orquestrador) registrando uma nova run do MLflow a cada execução, permitindo comparar a evolução do modelo ao longo do tempo.
- Promover a melhor run para um **Model Registry** do MLflow, versionando qual modelo está de fato servindo na API (Etapa 5) em produção.
- Adicionar métricas de monitoramento pós-deploy (ex.: taxa de conversão real observada em produção vs. estimada aqui), fechando o ciclo de MLOps com feedback contínuo.
